# zsad-stress-test: benchmark jobs on Kaggle

Thin wrapper around `eval/run_benchmark.py`. Normally submitted with `scripts/kaggle_submit.py`, which fills in the parameters cell. Code is cloned to /tmp so only `outputs/` is saved as notebook output.

In [ ]:
# parameters (overwritten by scripts/kaggle_submit.py)
BRANCH = 'main'
JOBS = [('anomalyvfm_dinov2', 'mvtec')]  # (method, dataset) pairs
EXTRA_ARGS = ''  # passed to every run_benchmark.py call
PRE_COMMANDS = []  # shell commands run after setup, before the jobs (e.g. training)

In [ ]:
!git clone -q -b {BRANCH} https://github.com/Addy0606/zsad-stress-test.git /tmp/zsad
%cd /tmp/zsad
!git log --oneline -1

In [ ]:
!pip install -q --no-cache-dir -r requirements-kaggle.txt
import torch

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
!python scripts/setup_third_party.py

In [ ]:
# find where Kaggle actually mounted the datasets and fix configs/paths/kaggle.yaml
!find /kaggle/input -maxdepth 3 -type d | head -40
!python scripts/check_paths.py --paths kaggle --search /kaggle/input --fix || true

In [ ]:
import subprocess
for cmd in PRE_COMMANDS:
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
import os
import subprocess

os.environ["HF_HOME"] = "/tmp/hf"
os.environ["TORCH_HOME"] = "/tmp/torch"
os.environ["CLIP_CACHE"] = "/tmp/clip"
failed = []
for method, dataset in JOBS:
    print(f"===== {method} on {dataset}", flush=True)
    cmd = f"python eval/run_benchmark.py --method {method} --dataset {dataset} --paths kaggle --out /kaggle/working/outputs {EXTRA_ARGS}"
    if subprocess.run(cmd, shell=True).returncode != 0:
        failed.append((method, dataset))
print("FAILED:", failed if failed else "none")